In [1]:
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv

repo = Path(find_dotenv(usecwd=True)).parent
date_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
orders = pd.read_csv(repo / "data" / "processed" / "orders_enriched.csv", parse_dates=date_cols)

print("shape:", orders.shape)

shape: (99441, 18)


In [2]:
# one row per person: their first order, delivered, with delivery info
m = orders[(orders["order_number"] == 1)
           & (orders["order_status"] == "delivered")
           & orders["is_late"].notna()
           & (orders["order_purchase_timestamp"] >= "2017-01-01")
           & (orders["order_purchase_timestamp"] < "2018-03-01")].copy()   # 6+ months to return

m["repeat"] = (m["person_total_orders"] > 1).astype(int)         # target: ordered again

features = ["order_value", "n_items", "delivery_days", "delay_days",
            "is_late", "review_score", "customer_state"]

print("people:", len(m), " repeat buyers:", m["repeat"].sum(), " rate %:", round(m["repeat"].mean() * 100, 2))
print(m[features].isna().sum())

people: 55227  repeat buyers: 2220  rate %: 4.02
order_value         0
n_items             0
delivery_days       0
delay_days          0
is_late             0
review_score      409
customer_state      0
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

X = m[features].copy()
X["is_late"] = X["is_late"].astype(int)
y = m["repeat"]

# 80/20 split; stratify keeps the 4% repeat share equal in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

num_cols = ["order_value", "n_items", "delivery_days", "delay_days", "is_late", "review_score"]
cat_cols = ["customer_state"]

prep = ColumnTransformer([
    ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

logit = Pipeline([("prep", prep),
                  ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logit.fit(X_train, y_train)

proba = logit.predict_proba(X_test)[:, 1]
print("train rows:", len(X_train), " test rows:", len(X_test))
print("repeat share train / test:", round(y_train.mean() * 100, 2), "/", round(y_test.mean() * 100, 2))
print("always-predict-no accuracy:", round((1 - y_test.mean()) * 100, 2))
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, logit.predict(X_test), digits=3))

train rows: 44181  test rows: 11046
repeat share train / test: 4.02 / 4.02
always-predict-no accuracy: 95.98
ROC-AUC: 0.539
              precision    recall  f1-score   support

           0      0.963     0.507     0.665     10602
           1      0.044     0.541     0.081       444

    accuracy                          0.509     11046
   macro avg      0.504     0.524     0.373     11046
weighted avg      0.927     0.509     0.641     11046



**Insight (baseline logistic regression):** On 55,227 first orders with at least 6 months to return (4.02% repeat buyers; 80/20 stratified split, 11,046 test rows), the model reaches ROC-AUC 0.539, barely above the 0.5 of random guessing. For repeat buyers, precision is 0.044 against a base rate of 4.02%, and recall is 0.541, reached by flagging about half of all customers. Accuracy is 50.9%, but a model that always predicts "no repeat" scores 95.98%, so accuracy is not a useful metric here.

**Caution:** Only 444 repeat buyers are in the test set, so the metrics are noisy. The features (order value, items, delivery, review score, state) describe the first order only, and I have no marketing, price or customer-history data.

**Business implication (hypothesis):** Repeat buying is hard to predict from first-order data. This fits the earlier tests, where delivery and review score barely moved the repeat rate.

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([("prep", prep),
               ("model", RandomForestClassifier(n_estimators=300, min_samples_leaf=50,
                                                class_weight="balanced_subsample",
                                                random_state=42, n_jobs=-1))])
rf.fit(X_train, y_train)

proba_rf = rf.predict_proba(X_test)[:, 1]   # probability of class 1 (repeat buyer)
print("Random forest ROC-AUC:", round(roc_auc_score(y_test, proba_rf), 3))
print("Logistic regression ROC-AUC:", round(roc_auc_score(y_test, proba), 3))

Random forest ROC-AUC: 0.533
Logistic regression ROC-AUC: 0.539


**Insight (model comparison):** Logistic regression (ROC-AUC 0.539) and a random forest (0.533) both rank repeat buyers only slightly better than random guessing (0.5). The 0.006 gap is smaller than the noise expected from 444 repeat buyers in the test set, so neither model is better. The features describe the first order only (value, items, delivery, review score, state).

**Business implication (hypothesis):** Repeat buying cannot be predicted from first-order data, so a scoring model built on these features would not help target customers. This fits the earlier tests, where delivery and review score had small or no links to repeat buying. Better signals would need data I do not have, such as marketing contact, prices, promotions or browsing behaviour.